# Laboratory 13 — Chemical potential

In this laboratory particles hop between boxes at random, and you find out what they end up
sharing. It is not the density. You measure the chemical potential three ways, watch it equalize
while the densities split, stack boxes into an atmosphere, turn one box into an adsorbing surface,
and finish by estimating the gas constant from (synthetic) osmotic-pressure readings.

The route: the model and its objects (Part 1); one mu, three ways (Part 2); the two-box exchange
(Part 3); two falsifying experiments (Part 4); the functional form of the split (Part 5); how
steady equal mu is (Part 6); the barometric column (Part 7); the adsorption site (Part 8);
osmosis and a measurement (Part 9); the automated checks (Part 10); and free exploration
(Part 11).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | $N$ identical, non-interacting particles on lattice-gas boxes; box $k$ has $M_k$ single-occupancy sites at energy $u_k$ |
| **Dynamics** | a random particle proposes a hop to a random site among all sites; a hop into another box's empty site is accepted with probability $\min(1, e^{-\Delta u/\kB T})$ |
| **Boundary** | closed to particles as a whole ($N$ exactly conserved); an implicit bath at $T$ supplies each hop's energy |
| **Ensemble** | canonical for the whole set; each box approximately grand canonical, the others its reservoir |
| **Ignored** | interactions beyond single occupancy, hop rates (time is counted in proposals), structure inside a box |
| **Valid when** | $M_k, N_k \gg 1$; the dilute formulas need $N_k \ll M_k$; the ideal-gas formulas need $n \ll n_Q$ |
| **Failure modes** | nearly full boxes, interacting particles (module 15), $n \to n_Q$ (module 17) |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import chemical, fundamental, potentials
from thermolab.constants import AMU, K_B, N_A, R_GAS
from thermolab.validation import (
    convergence_study,
    relative_error,
    scaling_exponent,
    seed_study,
)

# Every stochastic call below takes an explicit numpy Generator; nothing is seeded globally.

T = 300.0  # K
KT = K_B * T
print(f"k_B T at {T:.0f} K = {KT:.4e} J = {KT / 1.602176634e-19:.4f} eV")

## Part 1 — The objects

`thermolab.chemical` has three kinds of object.

| function | what it gives |
|---|---|
| `mu_from_entropy(relation, U, V, N)` | $-T(\partial S/\partial N)_{U,V}$ by central difference on any $S(U, V, N)$ |
| `mu_from_gibbs(relation, T, P, N)` | $(\partial G/\partial N)_{T,P}$ by central difference on module 10's $G$ |
| `ideal_gas_mu(n, T, m, u_ext)` | $\kB T\ln(n/n_Q) + u_{\mathrm{ext}}$ |
| `two_boxes`, `column` | sets of lattice-gas boxes |
| `particle_exchange_sim(boxes, counts, T, steps, rng)` | the hopping simulation; records each box's count |
| `exact_count_distribution(boxes, N, T)` | every split of two boxes, weighted exactly |
| `site_occupation(mu, T, epsilon)` | $1/(e^{(\varepsilon - \mu)/\kB T} + 1)$ |

A lattice-gas box of $M$ sites holding $N$ particles at site energy $u$ has chemical potential
$\mu = u + \kB T\ln[N/(M - N)]$, which for a dilute box is $u + \kB T\ln(N/M)$: an energy, plus an
entropic term that grows with the density.

### Predict

Commit to an answer for each of the module page's four predictions before going further:

1. Two boxes, B's floor raised, start at *equal* densities. Do particles flow? Which way?
2. Raise B's floor from $2$ to $4\,\kB T$. By what factor does the final density ratio change?
3. No floors; one box ten times larger; equal *numbers* to start. Do particles flow?
4. A bag of sugar solution in pure water, the wall passing only water. Which way does water move?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — One mu, three ways

Argon at $300\ \mathrm{K}$ and $1\ \mathrm{bar}$. Compute its chemical potential:

- as the slope $-T(\partial S/\partial N)_{U,V}$ of the Sackur–Tetrode $S(U, V, N)$ — with $T$ read
  off the same surface, so nothing but $S$ is used;
- as the slope $(\partial G/\partial N)_{T,P}$ of module 10's $G(T, P, N)$, built numerically from
  the same $S$;
- from the closed form $\kB T\ln(n/n_Q)$.

### Predict

Will the three agree? Will $\mu$ be positive or negative, and roughly how many $\kB T$?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
ARGON = 39.95 * AMU
PRESSURE = 1.0e5
N_GAS = 1.0e22
volume = N_GAS * KT / PRESSURE
energy = 1.5 * N_GAS * KT
argon = fundamental.monatomic_ideal_gas(ARGON)

mu_entropy = chemical.mu_from_entropy(argon, energy, volume, N_GAS)
mu_gibbs = chemical.mu_from_gibbs(argon, T, PRESSURE, N_GAS)
mu_closed = chemical.ideal_gas_mu(N_GAS / volume, T, ARGON)
g_per_particle = potentials.gibbs_from(argon, T, PRESSURE, N_GAS) / N_GAS
n_q = chemical.quantum_concentration(T, ARGON)

print(f"n = {N_GAS / volume:.3e} m^-3,  n_Q = {n_q:.3e} m^-3,  n/n_Q = {N_GAS / volume / n_q:.2e}")
print(f"-T dS/dN (entropy slope): {mu_entropy / KT:+.9f} k_B T")
print(f" dG/dN   (Gibbs slope):   {mu_gibbs / KT:+.9f} k_B T")
print(f" k_B T ln(n/n_Q):         {mu_closed / KT:+.9f} k_B T")
print(f" G / N:                   {g_per_particle / KT:+.9f} k_B T")
print(f"relative gaps: {relative_error(mu_entropy, mu_closed):.1e}, "
      f"{relative_error(mu_gibbs, mu_closed):.1e}, {relative_error(g_per_particle, mu_closed):.1e}")

The entropy route is a central difference in $N$, so its error should fall as the square of the
step. Measure the order — this is the first measurement of the laboratory.

In [ ]:
stencil = convergence_study(
    lambda k: chemical.mu_from_entropy(argon, energy, volume, N_GAS, dn=N_GAS / k),
    [4, 8, 16, 32, 64], mu_closed)
ratios = stencil.errors[:-1] / stencil.errors[1:]
print("dn = N/k for k =", stencil.refinements.astype(int).tolist())
print("relative errors:", ", ".join(f"{e:.2e}" for e in stencil.errors))
print(f"each halving divides the error by {ratios.min():.3f} to {ratios.max():.3f}")
print(f"observed order: {stencil.observed_order:.4f}")

The three agree, and $\mu$ is about $-16\,\kB T$: negative, because argon at $1\ \mathrm{bar}$ is ten
million times more dilute than its quantum concentration.

## Part 3 — Two boxes exchanging particles

Two boxes of $10^5$ sites each, $2000$ particles, box B's floor raised by $2\,\kB T$, every particle
starting in B. The dilute relaxation time $\tau$ — in proposed hops — sets how long to run.

### Predict

Where will the count in A settle? Sketch $\mu_A - \mu_B$ against time.

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
SITES, N_TOTAL, OFFSET = 100_000, 2000, 2.0
boxes = chemical.two_boxes(SITES, SITES, 0.0, OFFSET * KT)
tau = chemical.relaxation_steps(boxes, N_TOTAL, T)
n_steps = int(20 * tau)
print(f"relaxation time tau = {tau:.0f} proposed hops; running {n_steps} per run")

trace = chemical.particle_exchange_sim(boxes, [0, N_TOTAL], T, n_steps,
                                       np.random.default_rng(0), record_every=20)
exact = chemical.exact_count_distribution(boxes, N_TOTAL, T)
dilute = chemical.equilibrium_split(boxes, N_TOTAL, T)
mus = trace.chemical_potentials / KT

fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
top.plot(trace.steps / tau, trace.counts[:, 0], color="#2563eb", lw=1)
top.axhline(exact.mean, color="k", lw=1, label="exact equilibrium")
top.set_ylabel("N_A")
top.legend()
bottom.plot(trace.steps / tau, mus[:, 0] - mus[:, 1], color="#dc2626", lw=1)
bottom.axhline(0, color="k", lw=1)
bottom.set_ylabel("(mu_A - mu_B) / k_B T")
bottom.set_xlabel("t / tau")
plt.tight_layout()
plt.show()

settled = trace.tail(0.5)
print(f"second half: N_A = {settled[:, 0].mean():.1f}, N_B = {settled[:, 1].mean():.1f}")
print(f"exact: N_A = {exact.mean:.1f} +/- {exact.std:.1f};  dilute formula: {dilute[0]:.1f}")
densities = settled.mean(axis=0) / boxes.sites
print(f"density ratio n_A/n_B = {densities[0] / densities[1]:.2f}  (e^2 = {np.exp(2):.2f})")
half = len(trace.steps) // 2
late_gap = np.mean(mus[half:, 0] - mus[half:, 1])
print(f"mean of mu_A - mu_B over the second half: {late_gap:+.4f} k_B T")

One run is one sample. Repeat it under eight independent generators and compare the mean with the
exact distribution — a result that disagrees by more than about three standard errors would be a
bug, not physics.

In [ ]:
def settled_count(rng):
    run = chemical.particle_exchange_sim(boxes, [0, N_TOTAL], T, n_steps, rng, record_every=20)
    return float(run.tail(0.5)[:, 0].mean())


exchange_study = seed_study(settled_count, n_seeds=8)
print(f"N_A over 8 seeds: {exchange_study.mean:.1f} +/- {exchange_study.standard_error:.1f}")
print(f"exact mean:       {exact.mean:.1f}")
print("agrees within 3 standard errors:", exchange_study.agrees_with(exact.mean))

## Part 4 — Two falsifying experiments

**Against the concentration rule.** Start the same two boxes at *equal* densities, $1000$
particles each. If particles flowed down the concentration difference, nothing would happen.

**Against "mu is the energy per particle".** Take away the floors. Box A has $10^4$ sites, box B
$10^5$, and each starts with $1000$ particles: every particle has the same energy, zero, in either
box.

### Predict

For each experiment: does anything flow, and which way?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
equal_start = chemical.particle_exchange_sim(boxes, [1000, 1000], T, n_steps,
                                             np.random.default_rng(1), record_every=20)
start_mu = equal_start.chemical_potentials[0] / KT
end_counts = equal_start.tail(0.5).mean(axis=0)
print("equal densities, B's floor raised by 2 k_B T:")
print(f"  start: mu_A = {start_mu[0]:+.2f}, mu_B = {start_mu[1]:+.2f} k_B T")
print(f"  end:   N_A = {end_counts[0]:.0f}, N_B = {end_counts[1]:.0f}")

no_floors = chemical.two_boxes(10_000, 100_000, 0.0, 0.0)
tau_nf = chemical.relaxation_steps(no_floors, 2000, T)
same_energy = chemical.particle_exchange_sim(no_floors, [1000, 1000], T, int(20 * tau_nf),
                                             np.random.default_rng(2), record_every=20)
start_mu_nf = same_energy.chemical_potentials[0] / KT
end_nf = same_energy.tail(0.5).mean(axis=0)
exact_nf = chemical.exact_count_distribution(no_floors, 2000, T).mean
print("no floors, box A ten times smaller, equal numbers:")
print(f"  start: mu_A = {start_mu_nf[0]:+.2f}, mu_B = {start_mu_nf[1]:+.2f} k_B T")
print(f"  end:   N_A = {end_nf[0]:.0f} (exact {exact_nf:.1f}), N_B = {end_nf[1]:.0f}")
print(f"  densities: {end_nf[0] / 10_000:.4f} and {end_nf[1] / 100_000:.4f}")

# The falsifying experiments: particles flowed with no concentration difference at all, and
# with no energy difference at all -- each time from the higher mu to the lower.
assert start_mu[1] > start_mu[0] and end_counts[0] > 1500
assert start_mu_nf[0] > start_mu_nf[1] and end_nf[0] < 300

In the first experiment particles flowed from a box exactly as crowded as the other; in the
second, between boxes where every particle had the same energy. Each time they went from higher
$\mu$ to lower.

## Part 5 — The functional form

Sweep B's floor from $0$ to $3\,\kB T$ and plot $\ln(n_A/n_B)$ at equilibrium against the offset.

### Predict

What shape, and what slope?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
offsets = np.arange(0.0, 3.01, 0.5)
log_ratios, log_errors = [], []
for offset in offsets:
    swept = chemical.two_boxes(SITES, SITES, 0.0, offset * KT)
    steps = int(20 * chemical.relaxation_steps(swept, N_TOTAL, T))

    def log_ratio(rng, swept=swept, steps=steps):
        n_a = chemical.particle_exchange_sim(swept, [0, N_TOTAL], T, steps, rng,
                                             record_every=20).tail(0.5)[:, 0].mean()
        return float(np.log(n_a / (N_TOTAL - n_a)))

    study = seed_study(log_ratio, n_seeds=4, base_seed=100)
    log_ratios.append(study.mean)
    log_errors.append(study.standard_error)

(sweep_slope, sweep_intercept), sweep_cov = np.polyfit(offsets, log_ratios, 1, cov=True)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(offsets, log_ratios, yerr=log_errors, fmt="o", color="#2563eb")
ax.plot(offsets, offsets, "k--", label="slope 1: n_A/n_B = exp(Delta u / k_B T)")
ax.set_xlabel("(u_B - u_A) / k_B T")
ax.set_ylabel("ln(n_A / n_B)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"fitted slope: {sweep_slope:.4f} +/- {np.sqrt(sweep_cov[0, 0]):.4f} per k_B T")

Each extra $\kB T$ of offset multiplies the density ratio by $e$: $n_A/n_B = e^{(u_B - u_A)/\kB T}$,
equal $\mu$ in the dilute limit.

## Part 6 — How steady is equal mu?

At equilibrium $\mu_A - \mu_B$ jitters about zero. The exact distribution of the split gives its
standard deviation for any $N$ without a simulation; the prediction of problem 4 of the problem set
is $\kB T/\sqrt{Np(1-p)}$, with $p$ the fraction in A.

In [ ]:
sizes = [100, 1_000, 10_000, 100_000]
p_a = 1 / (1 + np.exp(-OFFSET))
spreads = []
for n in sizes:
    sized = chemical.two_boxes(100 * n, 100 * n, 0.0, OFFSET * KT)
    spreads.append(chemical.exact_count_distribution(sized, n, T).mu_difference_std() / KT)
    print(f"N = {n:>7,d}   std(mu_A - mu_B) = {spreads[-1]:.5f} k_B T   "
          f"(1/sqrt(N p (1-p)) = {1 / np.sqrt(n * p_a * (1 - p_a)):.5f})")
fluctuation_exponent = scaling_exponent(sizes, spreads)
print(f"fitted exponent: {fluctuation_exponent:.4f}  (the argument says -1/2)")

## Part 7 — The barometric column

Stack twelve boxes of $50\,000$ sites into a column, each layer $0.4\,\kB T$ above the one below —
as a layer of air sits $m g\,\Delta z$ above the one beneath it. Put all $2000$ particles in the
bottom layer and let them hop.

### Predict

What profile do they settle into? What is the same in every layer?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
LAYERS, LAYER_STEP = 12, 0.4
tower = chemical.column(LAYERS, 50_000, LAYER_STEP * KT)
start = [N_TOTAL] + [0] * (LAYERS - 1)


def column_run(rng):
    return chemical.particle_exchange_sim(tower, start, T, 400_000, rng, record_every=500)


column_runs = [column_run(np.random.default_rng(s)) for s in range(4)]
profile = np.mean([run.tail(0.5).mean(axis=0) for run in column_runs], axis=0)
layer_mu = np.mean([run.chemical_potentials[len(run.steps) // 2:].mean(axis=0)
                    for run in column_runs], axis=0) / KT
(column_slope, _), column_cov = np.polyfit(np.arange(10), np.log(profile[:10]), 1, cov=True)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
layers = np.arange(LAYERS)
left.barh(layers, profile, color="#2563eb", alpha=0.8, label="simulated")
left.plot(chemical.equilibrium_split(tower, N_TOTAL, T), layers, "k--",
          label="n(0) exp(-m g z / k_B T)")
left.set_xlabel("particles in layer")
left.set_ylabel("layer (height in units of k_B T / (0.4 m g))")
left.legend()
right.plot(layer_mu, layers, "o", color="#dc2626")
right.set_xlabel("mu / k_B T")
plt.tight_layout()
plt.show()

print(f"fitted decay per layer: {column_slope:.4f} +/- {np.sqrt(column_cov[0, 0]):.4f}"
      f"   (barometric formula: {-LAYER_STEP})")
print(f"spread of the layers' mu: {np.ptp(layer_mu):.3f} k_B T")

The same profile follows from $\mu$ alone. For real air — nitrogen at $288\ \mathrm{K}$ —
`barometric_profile` fixes $\mu$ at the ground and asks each height for the density that has the
same $\mu$ once $m g z$ is added. Compare it with module 11's Boltzmann factor $e^{-m g z/\kB T}$.

In [ ]:
NITROGEN = 28.0134 * AMU
heights = np.linspace(0.0, 30_000.0, 7)
from_mu = chemical.barometric_profile(heights, 288.0, NITROGEN, 2.5e25)
from_boltzmann = 2.5e25 * np.exp(-NITROGEN * chemical.STANDARD_GRAVITY * heights / (K_B * 288.0))
for z, a, b in zip(heights, from_mu, from_boltzmann, strict=True):
    print(f"z = {z / 1000:5.1f} km   constant mu: {a:.4e}   Boltzmann: {b:.4e}   "
          f"ratio {a / b:.15f}")
barometric_gap = float(np.max(np.abs(from_mu / from_boltzmann - 1)))

Two derivations, one atmosphere.

## Part 8 — The adsorption site

A surface of $200$ binding sites at $\varepsilon = -3\,\kB T$ exchanges particles with a reservoir of
$20\,000$ sites at energy $0$. Adding particles raises the reservoir's $\mu$. The grand-canonical
weight predicts that each surface site is filled with probability
$1/(e^{(\varepsilon - \mu)/\kB T} + 1)$ — whatever the reservoir's size.

### Predict

At what reservoir $\mu$ is half the surface covered?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
SURFACE, RESERVOIR, EPSILON = 200, 20_000, -3.0 * KT
totals = [20, 60, 150, 300, 600, 1000, 1600, 2400]
adsorption = []
for total in totals:
    pair = chemical.two_boxes(SURFACE, RESERVOIR, EPSILON, 0.0)
    mu_common, _ = chemical.occupation_equilibrium(pair, total, T)
    exact_cover = chemical.exact_count_distribution(pair, total, T).mean / SURFACE
    run = chemical.particle_exchange_sim(pair, [0, total], T, 300_000,
                                         np.random.default_rng(total), record_every=50)
    simulated = float(run.tail(0.5)[:, 0].mean() / SURFACE)
    formula = chemical.site_occupation(mu_common, T, EPSILON)
    adsorption.append((mu_common, exact_cover, formula, simulated))
    print(f"N = {total:5d}   mu = {mu_common / KT:+.3f} k_B T   exact {exact_cover:.4f}   "
          f"formula {formula:.4f}   simulated {simulated:.4f}")

mu_values = np.array([row[0] for row in adsorption]) / KT
grid = np.linspace(-8, 0, 300)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(grid, chemical.site_occupation(grid * KT, T, EPSILON), "k", label="1/(exp((eps-mu)/kT)+1)")
ax.plot(mu_values, [row[3] for row in adsorption], "o", color="#2563eb", label="simulated")
ax.axvline(EPSILON / KT, color="grey", ls=":")
ax.set_xlabel("mu / k_B T")
ax.set_ylabel("fraction of surface covered")
ax.legend()
plt.tight_layout()
plt.show()
occupation_gap = max(abs(row[1] - row[2]) for row in adsorption)

The surface responds to $\mu$ and nothing else. Written in terms of the gas pressure, the same
curve is the Langmuir isotherm $\theta = P/(P + P_0)$.

## Part 9 — Osmosis, and a measurement

A membrane passes water but not sugar. In the ideal-solution model, equal water chemical
potential across it requires $\Pi v = -\kB T\ln(1 - x)$, whose dilute limit is van 't Hoff's
$\Pi = n_s\kB T$. First compare the two; then fit synthetic readings.

> **About the readings.** The readings below are **synthetic**: generated by
> `chemical.osmosis_readings` from this model with a $3\%$ random error, standing in for a class's
> dialysis-tubing measurements. They can only confirm the model they came from. To use your own
> data, replace the two arrays.

In [ ]:
T_ROOM = 298.15
v_water = chemical.WATER_MOLECULAR_VOLUME
print("c [mol/L]   van 't Hoff [MPa]   -(kT/v) ln(1-x) [MPa]   difference")
for molar in (0.01, 0.1, 0.5, 1.0):
    solute = molar * 1e3 * N_A
    dilute_pi = chemical.osmotic_pressure(solute, T_ROOM)
    lattice_pi = chemical.lattice_osmotic_pressure(solute * v_water, v_water, T_ROOM)
    print(f"{molar:9.2f}   {dilute_pi / 1e6:17.4f}   {lattice_pi / 1e6:21.4f}   "
          f"{lattice_pi / dilute_pi - 1:.3%}")

concentrations = np.array([0.05, 0.1, 0.2, 0.3, 0.4, 0.5])  # mol/L
readings = chemical.osmosis_readings(concentrations, T_ROOM, 0.03, np.random.default_rng(13))

(rt_slope, rt_intercept), rt_cov = np.polyfit(concentrations * 1e3, readings, 1, cov=True)
rt_error = float(np.sqrt(rt_cov[0, 0]))
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(concentrations, readings / 1e6, "o", color="#2563eb", label="synthetic readings")
ax.plot(concentrations, (rt_slope * concentrations * 1e3 + rt_intercept) / 1e6, "k--", label="fit")
ax.set_xlabel("concentration [mol/L]")
ax.set_ylabel("osmotic pressure [MPa]")
ax.legend()
plt.tight_layout()
plt.show()
print(f"fitted slope R T = {rt_slope:.0f} +/- {rt_error:.0f} J/mol;  R T at {T_ROOM} K = "
      f"{R_GAS * T_ROOM:.0f} J/mol")
print(f"R estimated from sugar water: {rt_slope / T_ROOM:.2f} +/- {rt_error / T_ROOM:.2f} "
      "J/(mol K)")

## Part 10 — Automated checks

A calculation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
# 1. Three routes to mu agree (Part 2), and the entropy stencil is second order.
assert relative_error(mu_entropy, mu_closed) < 1e-6
assert relative_error(mu_gibbs, mu_closed) < 1e-6
assert relative_error(g_per_particle, mu_closed) < 1e-9
assert abs(stencil.observed_order - 2.0) < 0.05

# 2. The simulated split agrees with the exact distribution (Part 3).
assert exchange_study.agrees_with(exact.mean)

# 3. Each extra k_B T of offset multiplies the density ratio by e (Part 5).
assert abs(sweep_slope - 1.0) < 5 * np.sqrt(sweep_cov[0, 0]) + 0.02

# 4. Fluctuations of mu_A - mu_B fall as N^(-1/2) (Part 6).
assert abs(fluctuation_exponent + 0.5) < 0.01

# 5. The column is barometric, with one mu at every height (Part 7).
assert abs(column_slope + LAYER_STEP) < 0.01
assert np.ptp(layer_mu) < 0.1
assert barometric_gap < 1e-12

# 6. The one-site formula reproduces the exact coverage (Part 8).
assert occupation_gap < 1e-3

# 7. The fitted R T is within four standard errors of the true value (Part 9).
assert abs(rt_slope - R_GAS * T_ROOM) < 4 * rt_error

print("all checks passed")

## Part 11 — Explore it yourself

Set the floor offset, the particle number and the two boxes' sizes, choose how the particles
start, then press **Run Interact**. The left panel shows the count in each box against time; the
right panel shows the two chemical potentials. Three things worth trying:

1. Make B ten times larger than A with no offset, and start with every particle in B. Where does
   it settle, and why?
2. Pick an offset and a size ratio that exactly cancel, so that the dilute split is even. Which
   combination does it?
3. Fill the boxes to a few percent. How far does the settled count drift from the dilute formula,
   and does it drift from the exact one?

In [ ]:
import ipywidgets as widgets


def explore(offset=2.0, log10_particles=3.0, sites_a=100_000, sites_b=100_000, start="all in B"):
    n = int(round(10**log10_particles))
    pair = chemical.two_boxes(int(sites_a), int(sites_b), 0.0, offset * KT)
    if n >= pair.total_sites:
        print("more particles than sites")
        return
    initial = {"all in B": [0, n], "all in A": [n, 0], "equal numbers": [n // 2, n - n // 2]}[start]
    if initial[0] > sites_a or initial[1] > sites_b:
        print("the starting box is too small for that many particles")
        return
    steps = int(15 * chemical.relaxation_steps(pair, n, T))
    run = chemical.particle_exchange_sim(pair, initial, T, steps, np.random.default_rng(7),
                                         record_every=max(1, steps // 400))
    mus = run.chemical_potentials / KT
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.8))
    left.plot(run.steps, run.counts[:, 0], label="N_A")
    left.plot(run.steps, run.counts[:, 1], label="N_B")
    left.axhline(chemical.equilibrium_split(pair, n, T)[0], color="k", ls="--", lw=1)
    left.set_xlabel("proposed hops")
    left.legend()
    right.plot(run.steps, mus[:, 0], label="mu_A")
    right.plot(run.steps, mus[:, 1], label="mu_B")
    right.set_xlabel("proposed hops")
    right.set_ylabel("mu / k_B T")
    right.legend()
    plt.tight_layout()
    plt.show()
    final = run.tail(0.5).mean(axis=0)
    print(f"settled: N_A = {final[0]:.1f}, N_B = {final[1]:.1f};  dilute formula "
          f"{chemical.equilibrium_split(pair, n, T)[0]:.1f};  exact "
          f"{chemical.exact_count_distribution(pair, n, T).mean:.1f}")


widgets.interact_manual(
    explore,
    offset=widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.25, description="u_B - u_A"),
    log10_particles=widgets.FloatSlider(value=3.0, min=2.0, max=4.3, step=0.1,
                                        description="log10 N"),
    sites_a=widgets.IntSlider(value=100_000, min=1_000, max=200_000, step=1_000,
                              description="sites A"),
    sites_b=widgets.IntSlider(value=100_000, min=1_000, max=200_000, step=1_000,
                              description="sites B"),
    start=widgets.Dropdown(options=["all in B", "all in A", "equal numbers"], description="start"),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "13-chemical-potential.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. In Part 4 particles flowed between two boxes at equal densities, and between two boxes where
   every particle had the same energy. What, in each case, made one box's $\mu$ higher?
3. Part 7's column and module 11's Boltzmann factor give the same atmosphere. What does each
   derivation assume?
4. Why did the adsorbing surface not care how large the reservoir was?

**Your answers:**

1.
2.
3.
4.